# Object detection on old maps


Goal is to detect buildings, both inhabited and un-inhabited from old maps and produce a GIS-layers of existing buildings for separate time-periods.

Further goal is then to analyze how the buildings have survived through the years.


The project was originally created in TensorFlow and Keras in 2023, now ported to pytorch-platform.

## Create the dataset

In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd

Create the Dataset-class, simple transformation and Dataloader-helper-function

In [2]:
class ObjectDetectionDataset(Dataset):
    def __init__(self, csv_file, transform=None):
        self.base_path = "data/train_256px_extended/" #
        self.annotations = pd.read_csv(csv_file)
        self.transform = transform

        # Group annotations by image as there are 0-n boxes per image
        self.image_groups = self.annotations.groupby("filename")

        # Unique image paths
        self.image_paths = list(self.image_groups.groups.keys())

    def __len__(self): # Returns the length of the IMAGES in dataset, not boxes
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path =  self.image_paths[idx]
        rows = self.image_groups.get_group(image_path) # Get all bboxes for image

        # Load image
        image = Image.open(self.base_path + image_path).convert("RGB")

        # Create lists to store the boxes and their labels
        boxes = []
        labels = []

        for _, row in rows.iterrows():
            boxes.append([
                row["xmin"],
                row["ymin"],
                row["xmax"],
                row["ymax"]
            ])
            labels.append(row["class_num"])

        # Operate based on if boxes on image:
        if len(boxes)==0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0, ), dtype=torch.int64)
        else:
            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)

        # Create target:
        target = {"boxes": boxes, "labels": labels}

        if self.transform:
            image = self.transform(image)

        return image, target

In [3]:
# Simple transformation, i.e. converting to tensor. 
transform = transforms.Compose([
    transforms.ToTensor(),  # Converts to [0,1] and CHW
    # TODO: Placeholder for image augmentation; Note that only non-position-changing (brightness, colors etc) can be used easily
])

In [4]:
dataset = ObjectDetectionDataset(
    csv_file="data/train_256px_annotations_extended_cleaned_2026.csv",
    transform=transform
)

def collate_fn(batch):
    return tuple(zip(*batch))

dataloader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=collate_fn
)

In [5]:
images, targets = next(iter(dataloader))

In [6]:
print(images[0])


tensor([[[0.9961, 0.9961, 0.9961,  ..., 0.9961, 0.9961, 0.9961],
         [0.9961, 0.9961, 0.9961,  ..., 0.9961, 0.9961, 0.9961],
         [0.9961, 0.9961, 0.9961,  ..., 0.9961, 0.9961, 0.9961],
         ...,
         [0.6510, 0.6431, 0.6941,  ..., 1.0000, 1.0000, 1.0000],
         [0.7765, 0.7137, 0.6902,  ..., 0.7843, 0.8078, 0.9804],
         [0.9216, 0.8745, 0.8902,  ..., 0.6157, 0.6392, 0.9569]],

        [[1.0000, 1.0000, 1.0000,  ..., 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000,  ..., 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000,  ..., 1.0000, 1.0000, 1.0000],
         ...,
         [0.6980, 0.6902, 0.7373,  ..., 0.9294, 0.9137, 0.9490],
         [0.8196, 0.7490, 0.7294,  ..., 0.7137, 0.7216, 0.8863],
         [0.9569, 0.9098, 0.9176,  ..., 0.5451, 0.5647, 0.8706]],

        [[0.9373, 0.9373, 0.9373,  ..., 0.9686, 0.9686, 0.9686],
         [0.9373, 0.9373, 0.9373,  ..., 0.9686, 0.9686, 0.9686],
         [0.9373, 0.9373, 0.9373,  ..., 0.9686, 0.9686, 0.

In [10]:
images[0].shape

torch.Size([3, 256, 256])

In [ ]:
print(len(targets))
targets[0]

4


In [ ]:
# Print some example data
print(images[0].shape) # RGB image; 3 x 256 x 256
print(targets[0]['boxes']) # Number of boxes
print(targets[0]['labels'])

torch.Size([3, 256, 256])
tensor([[ 13.,  49.,  30.,  65.],
        [187.,  53., 209.,  69.],
        [176.,  91., 192., 114.],
        [189.,  91., 207., 112.],
        [182., 107., 204., 122.]])
tensor([1, 1, 1, 1, 2])


In [56]:
# Sanity checks
images, targets = next(iter(dataloader))
for i, t in enumerate(targets):
    print(f"Image {i}:")
    print(" Boxes shape:", t["boxes"].shape)
    print(" Labels shape:", t["labels"].shape)

Image 0:
 Boxes shape: torch.Size([4, 4])
 Labels shape: torch.Size([4])
Image 1:
 Boxes shape: torch.Size([1, 4])
 Labels shape: torch.Size([1])
Image 2:
 Boxes shape: torch.Size([3, 4])
 Labels shape: torch.Size([3])
Image 3:
 Boxes shape: torch.Size([4, 4])
 Labels shape: torch.Size([4])


## Model

In [ ]:
import torchvision.models.detection.fasterrcnn_resnet50_fpn

import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

In [ ]:

model = torchvision.models.detection.fasterrcnn_resnet50_fpn(
    pretrained=True,
    min_size=256,
    max_size=256
)


In [ ]:
num_classes = 3  # background + 2 object classes
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)


Parameter tuning - need to customize for the small objects, as they are only covering 1% of the image-area.

In [ ]:
from torchvision.models.detection.rpn import AnchorGenerator

anchor_generator = AnchorGenerator(
    sizes=((8, 16, 32, 64),),  # MUCH smaller than defaults
    aspect_ratios=((0.5, 1.0, 2.0),)
)

model.rpn.anchor_generator = anchor_generator

In [ ]:
# Treshold for the detection
model.rpn.fg_iou_thresh # defaults to 
# model.rpn.fg_iou_thresh = 0.7

In [ ]:
# How many samples per image? If objects are relatively rare (i.e. most is background), this should be incresed
model.rpn.batch_size_per_image # defaults to 
# model.rpn.batch_size_per_image = 256

In [ ]:
# Region-Of-Interest (ROI): Finetuning the anchored area:
# Setting to lower increases number of detections --> lift later to reduce false positives
model.roi_heads.score_thresh = 0.3

# Non-maximum-supression (nms): How much overlap is tolerated? Lower means clustered objects are outputted as single
model.roi_heads.nms_thresh = 0.5

In [ ]:
# How many objects to detect from image?
model.roi_heads.detections_per_img = 100